# Create FNP Awards (Foundation for Polish Science / Fundacja na rzecz Nauki Polskiej)

Creates Foundation for Polish Science awards from FNP's own laureates page,
https://www.fnp.org.pl/en/component/fnp_pages/page/about-laureates (two
server-rendered HTML tables; no export or API exists):

1. **Laureates of programmes financed by FNP's internal funds** (3,868 rows):
   FNP START stipends 1993-2026 (3,709), the FNP Prize 1992-2025 (124),
   FOR UKRAINE (14), Copernicus Polish-German award (10), Curie Polish-French
   award (5), Poland-U.S. award (3), Kolakowski fellowship (3). Year,
   laureate(s), institution, field. **No titles or amounts** are published
   for these; `display_name` is composed as `{programme} {year}: {laureate}`.
2. **FENG research projects** (149 rows, 2023-2026): FIRST TEAM, TEAM NET,
   MAB/IRAP, PoC and PRIME under the EU FENG 2021-2027 programme that FNP
   runs as intermediate body. Title, head researcher, beneficiary, call,
   amount in **PLN**.

**Filter:** everything on the page is research money (stipends, research
prizes, fellowships, research projects). MONOGRAPHS (publishing grants) is
not listed on the page and is not ingested.

**NOT covered:** FNP's 2007-2020 EU-funded programmes (TEAM, HOMING,
POWROTY, MAB, TEAM-TECH, TEAM NET, FIRST TEAM under POIG/POIR). FNP no
longer lists their laureates (archiwalna.fnp.org.pl has only competition PDFs).

**Prerequisites:**
- Run `scripts/local/fnp_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/fnp/fnp_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** FNP publishes no award number for these rows.
Citing works (`works?filter=grants.funder:F4320321042&select=grants`) write
agreement numbers such as `START 38.2016`, `TEAM/2016-2/17` or
`POIR.04.04.00-00-1AF0/16-00`, none of which appear on the site, so the key is
synthetic: `FNP-{programme}-{laureate page id}` where the row links a detail
page, else `FNP-{programme}-{year}-{SURNAME}-{GIVEN}` (4 homonym pairs get the
institution appended). Expect ~0 collapse onto existing citation stubs.

**Names:** the laureate table is "Surname Given"; the FENG table mixes both
orders and is resolved in the script with a given-name vocabulary learned
from the laureate table. Two-laureate awards carry both people in
`investigators` (lead = first listed).

**Dates:** year only; `start_date` = Jan 1 of the award year, no end date.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320321042`
- display_name / ror_id / doi: from `openalex.funders.funders` (Fundacja na rzecz Nauki Polskiej, ror 048zd9m77, doi 10.13039/501100001870)

**Priority:** `511` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fnp_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/fnp/fnp_projects.parquet`;

In [ ]:
%sql
SELECT source_table, programme, COUNT(*) as n FROM openalex.awards.fnp_raw GROUP BY 1, 2 ORDER BY n DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.fnp_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.fnp_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321042 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320321042;

## Step 2: Create FNP Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.fnp_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320321042  -- Fundacja na rzecz Nauki Polskiej (FNP)
),

people AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            COALESCE(from_json(people_json, 'ARRAY<STRUCT<name:STRING, given_name:STRING, family_name:STRING, institution:STRING>>'), ARRAY()),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(p.institution), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS investigators
    FROM openalex.awards.fnp_raw
),

named AS (
    SELECT funder_award_id,
           array_join(TRANSFORM(investigators, x -> CONCAT_WS(' ', x.given_name, x.family_name)), ' and ') AS laureates
    FROM people
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    -- FENG rows carry a project title; laureate rows get "{programme} {year}: {laureate(s)}".
    COALESCE(
        NULLIF(TRIM(g.title), ''),
        CONCAT(g.programme, ' ', g.year, ': ', NULLIF(n.laureates, ''))
    ) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'PLN' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    CASE WHEN g.call IS NOT NULL THEN CONCAT(g.programme, ' (call ', g.call, ')') ELSE g.programme END as funder_scheme,
    'fnp_laureates' as provenance,
    TRY_TO_DATE(CONCAT(TRY_CAST(g.year AS INT), '-01-01'), 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.fnp_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
JOIN named n ON n.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every funder_award_id is the synthetic FNP-{programme}-... key.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^FNP-[A-Z-]+-[0-9]' THEN 1 ELSE 0 END) as good_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^FNP-[A-Z-]+-[0-9]' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.fnp_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fnp_laureates' AND priority = 511;

-- Insert into openalex_awards_raw with priority.
-- Priority 511: direct-from-funder ingest of FNP's own laureate tables.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    511 as priority
FROM openalex.awards.fnp_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_fnp_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.fnp_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.fnp_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_pln
FROM openalex.awards.fnp_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.fnp_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.fnp_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Coverage (expect 100% title/PI/institution/start year; amount only on the 149 FENG rows, ~3%).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.fnp_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs for FNP (expect ~0: FNP publishes no award numbers).
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(n.id) as upgraded_by_fnp_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.fnp_awards n ON n.id = c.id
WHERE c.funder_id = 4320321042 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'fnp_laureates'
GROUP BY provenance, priority;